# GeneNetwork Data Investigation, Evaluation & Improvements Notebook
Author: Brian Muhia

Date: Sun Feb 25 2024

Connect this notebook to a kernel running on GeneNetwork's instance by running the following commands:

First, share a public key from a new ed25519 keypair. If you don't have one for Fahammu access, this can be generated by
```sh
ssh-keygen -t ed25519 -C "your-email"
```

After saving the file, share the `.pub` file generated to brian@fahamuai.com and we will add it to the GeneNetwork server's allow list.

Then, in your computer, find the file `~/.ssh/config` and add the following block:
```sh
Host genenetwork-azure-api.fahamuai.com
     User deploy
     HostName 20.124.120.32
     IdentityFile ~/.ssh/your_ssh_private_key
```

In the terminal:
```sh
$ eval "$(ssh-agent -s)"
$ ssh-add ~/.ssh/your_ssh_private_key
$ ssh -L8888:localhost:8888 genenetwork-azure-api.fahamuai.com
```

Once this is connected, the following command should give you a link to the running kernel, which you will need to copy.

```sh
$ jupyter notebook list
Currently running servers
http://localhost:8888/?token=350e4295c738582313de0a8aa338f76c60ba0b87d1083538 :: /opt/fahamu/wolfshead/priv/ice
```

In VSCode or Jupyter Lab, start a new `Python3` kernel for the notebook. After selecting "Existing Jupyter server", you can now copy (`Shift+Drag Cursor` then `Ctrl+Shift+C`) and paste this URL (including the token but nothing after that)

The next cell enables some features in jupyter that we need

In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [25]:
# this installs my fork of dspy, which I created to allow importation of datasets in json format.
#!pip install -U git+https://github.com/poppingtonic/dspy@add-from-json-data-loader

In [2]:
!pip install -U dspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.2/345.2 kB 1.6 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 5.1 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.3/454.3 kB 3.6 MB/s eta 0:00:00m eta 0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 152.9 kB/s eta 0:00:001m? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 1.1 MB/s eta 0:00:000:00:010:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 658.2 kB/s eta 0:00:00 kB/s eta 0:00:011
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 750.1 kB/s eta 0:00:001m810.6 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 1.1 MB/s eta 0:00:00.0 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.2/92.2 kB 684.5 kB/s eta 0:00:00m eta 0:00:016m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
!pip install fsspec

In [5]:
import os
import dspy
import json
import openai
from dspy.datasets import DataLoader
from dsp.utils import dotdict
from sqlalchemy import create_engine, text

ModuleNotFoundError: No module named 'fsspec'

### 1. The Data

The goal of this notebook is to explore and evaluate performance of `gpt-3.5-turbo` and `gpt-4` on 41 response objects from GeneNetwork's Digest API. We use them to develop an improvement to the Q&A pipeline using `DSPy`, for which we develop a few new modules that fit our database technology choices (`postgresql` and `pgvector`).

We collected these data during the 5 day period when they were testing the metadata system. They are saved as a json file using this SQL query in the GeneNetwork database _(hidden, click the ...)_:


```SQL
COPY(select json_agg(data) from tasks where task_id in ('FF6BD454EBC65E9507D680578A57C1D9',
'639273F37225EFFF8515AEA31D62DA89',
'AAC2C5C820FE495FD0C74F0ACDB4B79C',
'D5E9BCE533129E1142BECDC360FAA8DD',
'E1575CB86345672FCA5A9BF3A435E577',
'D4A123C699F7A17AD4D6F16AEEA61152',
'1783D393084B59E8898B1C1CD5BCD916',
'56D0E53C59A1E9B99CCBD0B2CF5AE7D5',
'3A595D6D74128BDE409E53AE9DD3D036',
'26F3A136782002293C1C5A7A9EC439D3',
'74C04F47335435E30B4B2805A2CF4ADB',
'CB01F4EDFEE30788452F4FBAE3E60A52',
'FFC12669C5D4955AF11EBC7B08649B3F',
'513210C71DB2E40E82827C2AE55E5195',
'9B84E150642DCCB0B51D014C32E55DCA',
'D5143E5788522FF60C44179D4B5D181C',
'2C9502925EAA752268F47F7D60383923',
'022654069D1829D34DB0A4E483E3CE66',
'426D630A4692B9260A61CA1152E57FF7',
'073B98C5443278C6F2A68092BDC15BA0',
'8596C9DF800592A255AC4A81DA53788E',
'13412C06635D9730486901119DBEAF72',
'A2BFA3E7F611CC3C02F40A423ABF1613',
'F5E94F882A6EC9D55DCFB2CC1649EF80',
'0CCE52BCCA0D84DCDEACB76D1FA871B4',
'F0E7D90B81884E34F4F68CBAD212CA0A',
'28A36292B2E34640FBCB72013597530A',
'E64FA0667CF4C1B98F0AF89CFF35FDBD',
'8CE95D351E60254AA6ECAD7F6E204CAB',
'E251B00669F6BC0F8B133F35545B464C',
'B9A6C04842B9FC45EBB4E2AB47ECA25C',
'8F31B6CC4BB03247564400565A4AC224',
'223AA2DD22659E4FC7ABD21768CB0D73',
'8A05836510C47DC42DB80152EBF262DF',
'48AD8F145E4E266D4CCF6F03CC63BD64',
'E983A551D2E773525F417DC653E422A2',
'15B35CD7E3D75D38E345F85728F35095',
'77B64E23D515601AF7C4525D91306E4C',
'04692819BEB702D114A09CC6DBFC939D',
'10ABB5180511A52BDDB2EFA3B6773B15',
'4E59EC6142BD4DC86E8EF72EF25B8075',
'F0374589E2EBEB4EC5A4B283959D0F87',
'39C12732C0CCA3BACCD768838B517CE4',
'5BE2A8B2E70C74BF03637607FE1AA56B')) TO '/tmp/demonstration_dataset_responses.json';
```

We load the data using the `from_json` function that I added in my fork of dspy.

In [26]:
dl = DataLoader()

In [27]:
digest_gn_dataset = dl.from_json("/opt/fahamu/wolfshead/priv/ice/demonstration_dataset_responses.json", 
                                 fields=["question", "answer", "context", "keywords", "data_source"], 
                                 input_keys=('question'))

In [28]:
digest_gn_dataset[20]

Example({'question': 'List genes or genotypes related to diabetes susceptibility in BXD mice', 'answer': 'The text does not provide specific information on genes or genotypes related to diabetes susceptibility in BXD mice.', 'context': [{'document_id': '29e232a4-a580-411d-83a3-7ff6a4e8f0ad', 'text': 'Results\\n\\nWe generated an F2 inter-cross between diabetes-resistant (B6) and diabetes-susceptible (BTBR) mouse strains, made genetically obese in response to the Lep ob mutation [24].The cross consisted of .500mice, evenly split between males and females.A comprehensive set of ,5000 genotype markers were used to genotype each F2 mouse (,2000 informative SNPs were used for analysis), and the expression levels of ,40 K transcripts (corresponding to 25,901 unique genes) were monitored in five tissues (adipose, liver, pancreatic islets, hypothalamus, and gastroc (gastrocnemius muscle)) that were harvested from each mouse at 10 weeks of age.In addition to gene expression, several key T2D-rel

We will first discuss some aspects of the data we will be looking at. The records we have include 'question', 'answer', 'keywords', 'context', 'data_source', among a few others not relevant for the NLP work that is our focus here.

The Keywords are supposed to follow a simple but strict syntax where we replace spaces in phrases and compound terms with `&` and every keyword should be separated with `|`. Being autogenerated by `gpt-4-turbo-preview` in an un-optimized pipeline, we see a few deviations from the instructions, and it is that pipeline that is being tested by our clients at the moment. The example below shows what a few of them would typically look like. The spaces between keywords are an instance of the model not following instructions, which we show here in order to test ways to make sure each space is a `&`.

In [8]:
digest_gn_dataset[10].keywords

['B6',
 'BTBR',
 'Lep ob',
 'SNPs',
 'adipose',
 'liver',
 'pancreatic islets',
 'hypothalamus',
 'gastrocnemius muscle',
 'T2D']

In [9]:
test_kw = ['human genome',
 'chromosomes',
 'DNA',
 'genes',
 'X chromosome',
 'Y chromosome',
 'autosomes',
 'mitochondrial DNA',
 'centromere',
 'chromatin']
t_kw="|".join(test_kw)

In [29]:
t_kw_cleaned = t_kw.replace(" ", "&")
t_kw_cleaned

'human&genome|chromosomes|DNA|genes|X&chromosome|Y&chromosome|autosomes|mitochondrial&DNA|centromere|chromatin'

We create dataset splits where the keywords with spaces in any of their terms are present in the test set, which we take to be "hard" examples for the model. The train set is selected by the opposite rule, including those keywords that correctly follow instructions.

In [30]:
test = [gn for gn in digest_gn_dataset if ' ' in '|'.join(gn.keywords)]

In [31]:
train = [gn for gn in digest_gn_dataset if ' ' not in '|'.join(gn.keywords)]

In [32]:
len(train)

25

Splitting the train set into `train_set` and `dev_set` which are `dspy.Prediction` objects where  we pre-select the input field `'question'`.

In [33]:
trainset = train[:17]
trainset = [gn.with_inputs("question") for gn in trainset]

devset = train[17:25]
devset = [gn.with_inputs("question") for gn in devset]


In [34]:
dev_set = dspy.Prediction(passages=devset).with_inputs('question')

In [35]:
train_set = dspy.Prediction(passages=trainset).with_inputs('question')

In [36]:
# We'll rely on turbo for everything except bootstrapping CoT demos:
api_key = os.getenv("OPENAI_API_KEY")
turbo = dspy.OpenAI(model='gpt-3.5-turbo-1106', max_tokens=250, model_type='chat', api_key=api_key)

In [37]:
len(devset)

8

In [38]:
devset[6]

Example({'question': 'List genes related to diabetes and their associated phenotypes.', 'answer': '1. PPARG and KCNJ11: These genes are involved in both monogenic and multifactorial forms of diabetes and encode the targets of proven diabetes drugs.\\n2. Leptin gene polymorphisms: These are associated with blood leptin levels and diet success in the extremely obese.\\n3. Glucokinase, IRS2, and PPAR␥: In knockout mice, these genes have been linked to the effect of different dietary fats on insulin resistance, ␤-cell hyperplasia, overt diabetes, and arterial hypertension.\\n4. HHEX, THADA, PPARG, KCNJ11: Disrupting orthologs of these T2D candidate genes led to sucrose-dependent toxicity in a Drosophila model.\\n5. dHHEX (CG7056): Fat-body-specific loss of this gene led to increased hemolymph glucose and reduced insulin sensitivity in Drosophila.\\n6. Pdk4, Adipoq, Scd, Pik3r1, Socs2: These genes monitor important hallmarks of T2DM, such as the strong relationship between obesity and insul

In [39]:
devset[6].context

[{'document_id': 'f9b65334-56b7-43e9-9fda-b778c18c1c67',
  'text': '\\n\\nGenomic information associated with Type 2 diabetes.'},
 {'document_id': '50c72e55-b5fe-42a6-b837-64c28620a4c0',
  'text': '\\n\\nGenetic determinants of diabetes and metabolic syndromes.'},
 {'document_id': '0da4d3d4-10d5-4a58-9e50-c1fa0b414427',
  'text': '\\n\\nOf course, none of this should come as a surprise.Once a gene has been shown to harbor one variant associated with, or causal for, a diabetes-like phenotype, it becomes far more likely that other nearby variants (provided they exert some effect on the expression and/or function of the gene) will also have a detectable phenotypic effect.By the same token, the genotype-phenotype relationships revealed by these gene discovery efforts highlight the pathways involved as prime candidates for beneficial therapeutic or preventative manipulation, a view reinforced by the fact that at least two of the genes involved in both monogenic and multifactorial forms of d

### 1.5 Handling Retrieval For Digest
We're storing data in postgresql, a standard choice for open source databases, and that offers us our own set of vector database options, out of which we chose `pgvector`. This comes through the [`pgvector-python`](https://github.com/pgvector/pgvector-python) library and a `psycopg2` connection. Retrieval happens through the use of a small SQL query with a custom registered `pgvector` operator `<->` for computing nearest neighbours, which is how we perform ranking based on the similarity between the user's question and a paragraph in the database.

The first exercise will be to implement a `dspy.Retrieve` class that connects to our postgresql database, registers the vector type with psycopg2 and opens an OpenAI client to compute query embeddings.

It uses the OpenAI client to compute the user's query embeddings, and psycopg2 to query the database using the embeddings in a raw SQL query, to find related paragraphs. Then, later in the notebook we will integrate an answering pipeline and a keyword generation pipeline. This will complete a thorough introduction to retrieval augmented generation `(RAG)` for the `Digest API`.

In [42]:
from typing import Optional
from pgvector.psycopg2 import register_vector
import psycopg2
from psycopg2 import sql

class PgVectorRM(dspy.Retrieve):
    """
    Implements a retriever that (as the name suggests) uses pgvector to retrieve passages,
    using a raw SQL query and a postgresql connection managed by psycopg2.

    It needs to register the pgvector extension with the psycopg2 connection

    Returns a list of dspy.Example objects
    """
    def __init__(self, k=20):
        """
        k = 20 is the number of paragraphs to retrieve
        """
        openai.api_key = os.environ.get("OPENAI_API_KEY", None)
        self.client = openai.OpenAI()
        
        # The DATABASE_URL stored in env uses Ecto's protocol, since the GeneNetwork API runs on Elixir.
        # we replace ecto with postgresql, to fit with psycopg2's DSN syntax.
        db_url=os.getenv("DATABASE_URL").replace('ecto', 'postgresql')
        self.conn = psycopg2.connect(db_url)
        register_vector(self.conn)

        super().__init__(k=k)

    def forward(self, query: str, k: Optional[int]=20):
        """Given a question, this returns a list of dspy.Example objects"""
        # Embed query
        query_embedding = self.client.embeddings.create(
            model="text-embedding-ada-002",
            input=query,
            encoding_format="float"
        ).data[0].embedding

        related_paragraphs = []
        passages = []

        with self.conn as conn:
            with conn.cursor() as cur:
                cur.execute(
                    sql.SQL('SELECT text, document_id FROM paragraphs ORDER BY embedding <-> %s::vector LIMIT %s'),
                    [query_embedding, self.k])
                rows = cur.fetchall()
                for row in rows:
                    related_paragraphs.append(dspy.Example(long_text=row[0], document_id=row[1]))
        # Return Prediction
        # passages.extend(dspy.Example(long_text=d["text"], document_id=d["document_id"]) for d in related_paragraphs)
        return related_paragraphs

We use `gpt-4` to give us evaluation metrics, and `gpt-3.5-turbo` to perform the tasks. The goal is to `compile` the pipeline we develop based on these metrics. Compilation is a series of steps that repeatedly test and improve variants of the instructions we give, checking the responses against the metrics until performance improves. Our targets will be to improve on cost, detail, faithfulness, accuracy of answers, and correctness of keywords based on the user's question and the retrieved context. 

The context retriever is the class `PgVectorRM` defined above, which returns 20 paragraphs.

In [45]:
metricLM = dspy.OpenAI(model='gpt-4', max_tokens=1000, model_type='chat')
llm = dspy.OpenAI(model='gpt-3.5-turbo', model_type='chat')

dspy.settings.configure(lm=llm, rm=PgVectorRM(k=20))


In [43]:
devset[6].question

'List genes related to diabetes and their associated phenotypes.'

In [57]:
test_passages = dspy.Retrieve()(devset[6].question)
# test_passages

In [58]:
test_passages

Prediction(
    passages=['\n\nGenomic information associated with Type 2 diabetes.', '\n\nGenetic determinants of diabetes and metabolic syndromes.', '\n\nOf course, none of this should come as a surprise.Once a gene has been shown to harbor one variant associated with, or causal for, a diabetes-like phenotype, it becomes far more likely that other nearby variants (provided they exert some effect on the expression and/or function of the gene) will also have a detectable phenotypic effect.By the same token, the genotype-phenotype relationships revealed by these gene discovery efforts highlight the pathways involved as prime candidates for beneficial therapeutic or preventative manipulation, a view reinforced by the fact that at least two of the genes involved in both monogenic and multifactorial forms of diabetes (PPARG, KCNJ11) encode the targets of proven diabetes drugs.', '\n\nDiabetes is a genetically complex multifactorial disease that requires sophisticated consideration of multi

In [59]:
test_passages.passages

['\n\nGenomic information associated with Type 2 diabetes.',
 '\n\nGenetic determinants of diabetes and metabolic syndromes.',
 '\n\nOf course, none of this should come as a surprise.Once a gene has been shown to harbor one variant associated with, or causal for, a diabetes-like phenotype, it becomes far more likely that other nearby variants (provided they exert some effect on the expression and/or function of the gene) will also have a detectable phenotypic effect.By the same token, the genotype-phenotype relationships revealed by these gene discovery efforts highlight the pathways involved as prime candidates for beneficial therapeutic or preventative manipulation, a view reinforced by the fact that at least two of the genes involved in both monogenic and multifactorial forms of diabetes (PPARG, KCNJ11) encode the targets of proven diabetes drugs.',
 '\n\nDiabetes is a genetically complex multifactorial disease that requires sophisticated consideration of multigenic and phenotypic i

Now that we have looked at the data and retrieval methods, the next major goal of this notebook is to  evaluate a run from the previous pipeline i.e. by taking the context, question and answer then using GPT-4 to design a metric that evaluates the quality of a generated answer to a question across multiple criteria, as well as the quality of the keywords. That becomes the baseline "gold" standard.

In [46]:
# Signature for LLM assessments
class Assess(dspy.Signature):
    """Assess the quality of an answer to a question."""

    context = dspy.InputField(desc="The context for answering the question.")
    assessed_question = dspy.InputField(desc="The evaluation criterion.")
    assessed_answer = dspy.InputField(desc="The answer to the question.")
    assessment_answer = dspy.OutputField(desc="A rating between 1 and 5. Only output the rating and nothing else.")

In [47]:
# Signature for LLM assessments for keyword generation
class AssessKeyword(dspy.Signature):
    ("""Assess the quality of a list of keywords generated to search metadata related to a question and context."""
     """The keywords generated should be separated by `|` characters, and any spaces should be replaced with the `&` character""")

    context = dspy.InputField(desc="The context for answering the question.")
    assessed_question = dspy.InputField(desc="The evaluation criterion.")
    assessed_keywords = dspy.InputField(desc="The keywords generated for the question.")
    assessment_answer = dspy.OutputField(desc="A rating between 1 and 5. Only output the rating and nothing else.")

## 2. LLM Metrics

Define a Metric for performance

In [48]:
def llm_gold_metric(gold, trace=None):
    """Evaluation metric for use with context from a static json dataset"""
    predicted_answer = gold.answer
    question = gold.question
    keywords = "|".join(gold.keywords)

    print(f"Test Question: {question}")
    print(f"Predicted answer: {predicted_answer}")

    detail = "Is the assessed answer detailed?"
    faithful = "Is the assessed text grounded in the context? Say 1 if it includes significant facts not in the context and 5 if it is grounded in the context."
    overall = f"Based on the context, please rate how well this answer answers the question, `{question}` \n Answer: `{predicted_answer}`"
    keyword = f"Do the keywords contain any spaces? `{keywords}`"

    with dspy.context(lm=metricLM):
        context = [dspy.Example(context=d["text"]) for d in gold.context]
        detail = dspy.ChainOfThought(Assess)(context=context, assessed_question=detail, assessed_answer=predicted_answer)
        faithful = dspy.ChainOfThought(Assess)(context=context, assessed_question=faithful, assessed_answer=predicted_answer)
        overall = dspy.ChainOfThought(Assess)(context=context, assessed_question=overall, assessed_answer=predicted_answer)
        keyword = dspy.ChainOfThought(AssessKeyword)(context=context, assessed_question=keyword, assessed_answer=predicted_answer, assessed_keywords=keywords)

    print(f"Faithful: {faithful.assessment_answer}")
    print(f"Detail: {detail.assessment_answer}")
    print(f"Overall: {overall.assessment_answer}")
    print(f"Keywords: {keyword.assessment_answer}")

    total = float(detail.assessment_answer) + float(faithful.assessment_answer) + float(overall.assessment_answer) + float(keyword.assessment_answer)
    return total / 5.0    

In [49]:
def llm_metric(gold, pred, trace=None):
    """Metric for use with the full retrieval stack, so context is dynamically retrieved for each question."""
    predicted_answer = pred.answer
    question = gold.question
    keywords = "|".join(pred.keywords)

    print(f"Test Question: {question}")
    print(f"Predicted answer: {predicted_answer}")
    print(f"Keyword string: {keywords}")

    detail = "Is the assessed answer detailed?"
    faithful = "Is the assessed text grounded in the context? Say 1 if it includes significant facts not in the context and 5 if it is grounded in the context."
    overall = f"Based on the context, please rate how well this answer answers the question, `{question}` \n Answer: `{predicted_answer}`"

    with dspy.context(lm=metricLM):
        context = PgVectorRM(k=20)(question).passages
        detail = dspy.ChainOfThought(Assess)(context="N/A", assessed_question=detail, assessed_answer=predicted_answer)
        faithful = dspy.ChainOfThought(Assess)(context=context, assessed_question=faithful, assessed_answer=predicted_answer)
        overall = dspy.ChainOfThought(Assess)(context=context, assessed_question=overall, assessed_answer=predicted_answer)

    print(f"Faithful: {faithful.assessment_answer}")
    print(f"Detail: {detail.assessment_answer}")
    print(f"Overall: {overall.assessment_answer}")

    total = float(detail.assessment_answer) + float(faithful.assessment_answer) + float(overall.assessment_answer)
    return total / 5.0    

### Inspect the metric

In [50]:
example = devset[6]

In [51]:
type(llm_gold_metric(example))

Test Question: List genes related to diabetes and their associated phenotypes.
Predicted answer: 1. PPARG and KCNJ11: These genes are involved in both monogenic and multifactorial forms of diabetes and encode the targets of proven diabetes drugs.\n2. Leptin gene polymorphisms: These are associated with blood leptin levels and diet success in the extremely obese.\n3. Glucokinase, IRS2, and PPAR␥: In knockout mice, these genes have been linked to the effect of different dietary fats on insulin resistance, ␤-cell hyperplasia, overt diabetes, and arterial hypertension.\n4. HHEX, THADA, PPARG, KCNJ11: Disrupting orthologs of these T2D candidate genes led to sucrose-dependent toxicity in a Drosophila model.\n5. dHHEX (CG7056): Fat-body-specific loss of this gene led to increased hemolymph glucose and reduced insulin sensitivity in Drosophila.\n6. Pdk4, Adipoq, Scd, Pik3r1, Socs2: These genes monitor important hallmarks of T2DM, such as the strong relationship between obesity and insulin resi

float

In [52]:
example_2 = devset[4]
llm_gold_metric(example_2)

Test Question: Which genes are related to increased hemolymph glucose?
Predicted answer: The genes related to increased glucose levels include MTNR1B, G6PC2, GCK, PANK1, Fgf15, Gm17685, Pla2g4a, Ptgs2, ADCY5, MADD, ADRA2A, CRY2, FADS1, GLIS3, SLC2A2, PROX1, C2CD4B, IGF1, TCF7L2, HNF1B, CTBP1, LEP, HHEX, TCF7L2, HNF1B, KCNQ1, NOTCH2, TCF7L2, THADA, TSPAN8, WFS1, HNF1B, IRS1, KCNJ11, NOTCH2, WFS1, GCK, GIGYF1, G6PC2, HNF1A, TNRC6B, and PDX1.
Faithful: 5
Detail: 5
Overall: 5
Keywords: 5


4.0

In [94]:
metricLM.inspect_history(n=2)





Assess the quality of an answer to a question.

---

Follow the following format.

Context: The context for answering the question.

Assessed Question: The evaluation criterion.

Assessed Answer: The answer to the question.

Reasoning: Let's think step by step in order to ${produce the assessment_answer}. We ...

Assessment Answer: A rating between 1 and 5. Only output the rating and nothing else.

---

Context:
[1] «Example({'context': 'Metadata is always public, even if the data are restricted or\\nremoved for privacy issues (‘F’, ‘A’). This metadata is offered at three levels, extensively supporting the\\n‘I’ and ‘R’ FAIR principles: 1) data citation metadata, which maps to DataCite schema or Dublin Core\\nTerms, 2) domain-speciﬁc metadata, which when possible maps to metadata standards used within a\\nscientiﬁc domain, and 3) ﬁle-level metadata, which can be deep and extensive for tabular data ﬁles\\n(including column-level metadata).'}) (input_keys=None)»
[2] «Example({'contex

In [28]:
len(metricLM.history)

0

In [92]:
# for gn_example in digest_gn_dataset:
#     llm_gold_metric(gn_example)

Test Question: chromosomes
Predicted answer: Chromosomes are threadlike structures where genes are arranged and lined up. They are made up of DNA and supporting proteins. Humans have 23 pairs of chromosomes, including one pair that determines sex. Each chromosome pair is inherited from the parents. The full set of chromosomes is collectively called the genome. The organization and structure of chromosomes are crucial for gene regulation and control of gene expression programs. Changes in the structure of DNA can sometimes cause harm to the body.
Faithful: 5
Detail: 5
Overall: 5
Keywords: 1
Test Question: what genes are involved in the aging process
Predicted answer: Several genes are involved in the aging process. These include BAZ2B, HMGB4, NOC2L, RAI1, SIK1, SMARCA2, SPZ1, TBP, TRIP13, and ZKSCAN1 which regulate transcription. DBH, TPO, and LSS genes are involved in the synthesis of hormones. GPER binds estrogen and HCRTR2 binds orexin-A and orexin-B neuropeptid hormones. ATG2A, NEDD

InvalidRequestError: This model's maximum context length is 8192 tokens. However, you requested 8305 tokens (7305 in the messages, 1000 in the completion). Please reduce the length of the messages or completion.

### The DSPy Programming Model

First we initialize the `GenerateAnswer`, `GenerateKeywords`, and `GenerateSearchQuery` signatures.

Then we will compose a `dspy.Module` consisting of:

+ `Retrieve`
+ `GenerateAnswer`

The DSPy programming model is one of the most powerful aspects of DSPy. Through it, we get:

+ An intuiitive interface to compose prompts into programs
+ A clean way to organize prompts into Signatures.
+ Structured output parsing with `dspy.OutputField`
+ Built-in prompt extensions such as `ChainOfThought`, `ReAct`, and more.

In [53]:
class GenerateAnswer(dspy.Signature):
    """Answers questions based on the context."""
    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="the best answer to the $question. it should be grounded in the context, detailed and helpful.")
    # keywords = dspy.OutputField(desc="10 keywords relevant for metadata search. They are for use in postgres, so each keyword should be separated by `|` and each multi-word phrase should replace the space character with `&`. ONLY the keywords.")

In [54]:
class GenerateKeywords(dspy.Signature):
    ("""Generates keywords based on the question and the context."""
     """Identify the named entities and generate keywords for use in postgres full text search""")
    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    keywords = dspy.OutputField(desc="10 keywords relevant for metadata search. Each keyword should be separated by `|` and each multi-word phrase should replace the space character with `&`. ONLY the keywords.")

In [65]:
class GenerateSearchQuery(dspy.Signature):
    """Write a simple declarative search query that will help answer a complex question"""
    context = dspy.InputField(desc="may contain relevant facts")
    question = dspy.InputField()
    search_query = dspy.OutputField(desc="based on the context, it should be different from the question, and should assist with answering it in more detail")

In [66]:
test_passages

Prediction(
    passages=['\n\nGenomic information associated with Type 2 diabetes.', '\n\nGenetic determinants of diabetes and metabolic syndromes.', '\n\nOf course, none of this should come as a surprise.Once a gene has been shown to harbor one variant associated with, or causal for, a diabetes-like phenotype, it becomes far more likely that other nearby variants (provided they exert some effect on the expression and/or function of the gene) will also have a detectable phenotypic effect.By the same token, the genotype-phenotype relationships revealed by these gene discovery efforts highlight the pathways involved as prime candidates for beneficial therapeutic or preventative manipulation, a view reinforced by the fact that at least two of the genes involved in both monogenic and multifactorial forms of diabetes (PPARG, KCNJ11) encode the targets of proven diabetes drugs.', '\n\nDiabetes is a genetically complex multifactorial disease that requires sophisticated consideration of multi

In [67]:
devset[5].question

'List genes related to diabetes'

A good example of the modularity offered by `DSPy`: combining `GenerateSearchQuery`, `dspy.Retrieve`, and our data object from the beginning of the notebook.

In [68]:
query_gen = dspy.Predict(GenerateSearchQuery)
query_gen(context=dspy.Retrieve()(devset[5].question).passages, question=devset[5].question)

Prediction(
    search_query='Genes associated with Type 2 diabetes mellitus'
)

### Digest
The `dspy.Module` Digest brings together everything into a simple interface for Q&A.

In [91]:
class Digest(dspy.Module):
    """
    Retrieves paragraphs for Q&A from the GeneNetwork database,
    and generates keywords suitable for metadata search, based on the question and
    retrieved paragraphs."""
    def __init__(self, num_passages=20):
        super().__init__()

        self.retrieve = dspy.Retrieve(k=20)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)
        self.generate_keywords = dspy.ChainOfThought(GenerateKeywords)
    
    def forward(self, question):
        context = self.retrieve(question).passages
        prediction = self.generate_answer(context=context, question=question)
        keywords_pred = self.generate_keywords(context=context, question=question)
        return dspy.Prediction(answer=prediction.answer, keywords=keywords_pred.keywords)

Similar to `Digest` above, this is a way to start performing the same function as the `amplify` pipeline in the `Digest v2` `ice_api`.

In [72]:
class AmplifyMultiHop(dspy.Module):
    """
    Performs one extra hop to generate a new search query that
    can help with providing a more detailed response"""
    def __init__(self, passages_per_hop=10):
        self.retrieve = PgVectorRM(k=passages_per_hop)
        self.generate_query = GenerateSearchQuery()
        # self.generate_query = dspy.ChainOfThought("context, question -> search_query")
        self.generate_answer = dspy.ChainOfThought("context, question -> answer")
    
    def forward(self, question):
        context = []
        for hop in range(2):
            query = self.generate_query(context=context, question=question).search_query
            context += self.retrieve(query).passages
        
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(answer=prediction.answer)

### Defining Evaluate

In [87]:
from dspy.evaluate.evaluate import Evaluate

In [88]:
dev_set

Prediction(
    passages=[Example({'question': 'List genotypes for diabetes susceptibility in BXD mice.', 'answer': 'The background text does not provide specific information on genotypes for diabetes susceptibility in BXD mice.', 'context': [{'document_id': '29e232a4-a580-411d-83a3-7ff6a4e8f0ad', 'text': 'Results\\n\\nWe generated an F2 inter-cross between diabetes-resistant (B6) and diabetes-susceptible (BTBR) mouse strains, made genetically obese in response to the Lep ob mutation [24].The cross consisted of .500mice, evenly split between males and females.A comprehensive set of ,5000 genotype markers were used to genotype each F2 mouse (,2000 informative SNPs were used for analysis), and the expression levels of ,40 K transcripts (corresponding to 25,901 unique genes) were monitored in five tissues (adipose, liver, pancreatic islets, hypothalamus, and gastroc (gastrocnemius muscle)) that were harvested from each mouse at 10 weeks of age.In addition to gene expression, several key T

In [89]:
evaluate = Evaluate(devset=dev_set, num_threads=1, display_progress=True, display_table=5)

In [232]:
devset[5].question

'List genes related to diabetes'

In [78]:
dspy.ChainOfThought(GenerateAnswer)(question=devset[5].question, context=dspy.Retrieve()(devset[5].question).passages)

Prediction(
    rationale='produce the answer. We need to look at the various studies and information provided in the context to identify genes related to diabetes.',
    answer='Some of the genes related to diabetes mentioned in the context include TCF7L2, PPARG, FTO, KCNJ11, NOTCH2, WFS1, CDKAL1, IGF2BP2, SLC30A8, JAZF1, HHEX, HNF1A, HNF4A, IPF1, and Pdx-1.'
)

In [79]:
dspy.ChainOfThought(GenerateKeywords)(question=devset[5].question, context=dspy.Retrieve()(devset[5].question).passages)

Prediction(
    rationale='produce the keywords. We will identify the named entities related to genes in the context and extract them as keywords for metadata search.',
    keywords='TCF7L2|PPARG|FTO|KCNJ11|NOTCH2|WFS1|CDKAL1|IGF2BP2|SLC30A8|JAZF1|HHEX|Pdk4|Adipoq|Scd|Pik3r1|Socs2|HNF1A|HNF4A|IPF1|Pdx-1|PPARγ2|ACE|MTHR|FABP2|FTO|HNF4A|IPF1|Pdx-1|H'
)

Testing retrieval

In [80]:
dspy.Retrieve()("Which genes are related to increased hemolymph glucose?")

Prediction(
    passages=['\n\nA number of recent GWASs have identified genes associated with glucose-related phenotypes: MTNR1B , G6PC2 and GCK for fasting plasma glucose [43][44][45] , PANK1 for fasting plasma insulin [43] .It would be highly relevant to further investigate whether these genes interact with environmental factors such as diet to influence glucose homeostasis-related phenotypes.', 'We found that 65\nof the 128 genes present in the Chr7 locus were expressed in\nthe hypothalamus.  The hypothalamic expression level of 11\ngenes was significantly correlated to the glucagon trait (Table\n1).  Fgf15 mRNA was the most significantly, and negatively,\ncorrelated to the glucagon phenotype (r = -0.571; p = 2.73 3\n10-4) (Table 1).  The second gene in this table, which shows a\nsimilar high correlation with the trait as Fgf15 is Gm17685 (r =\n\x030562; p = 3.58 3 10\x034).', '\n\nAssociations of iron-related loci with glycemic traits', 'All genes under the four significant or sugg

In [81]:
dspy.settings.rm

In [37]:
#!pip install ipdb

### dspy.ReAct (Experimental) 
The next 2 cells are a test of the ReAct pipeline which still has a few bugs. Ignore this.

In [82]:
class DebugReAct(dspy.ReAct):
    """Implemented to debug the ReAct pipeline which responds with an empty answer. So far I followed
    the error to this function, which has debug traces to investigate the issue."""
    def act(self, output, hop):
        try:
            action = output[f"Action_{hop+1}"]
            action_name, action_val = action.strip().split('\n')[0].split('[', 1)
            action_val = action_val.rsplit(']', 1)[0]

            if action_name == 'Finish': return action_val

            output[f"Observation_{hop+1}"] = self.tools[action_name](action_val).passages

        except Exception as e:
            import ipdb; ipdb.set_trace()
            output[f"Observation_{hop+1}"] = "Failed to parse action. Bad formatting or incorrect action name."
        

In [49]:
DebugReAct(GenerateAnswer, tools=[dspy.settings.rm])(question="Which genes are related to increased hemolymph glucose?")

> /tmp/ipykernel_2113022/1195860776.py(14)act()
     13             import ipdb; ipdb.set_trace()
---> 14             output[f"Observation_{hop+1}"] = "Failed to parse action. Bad formatting or incorrect action name."
     15 



--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> /tmp/ipykernel_2113022/1195860776.py(14)act()
     13             import ipdb; ipdb.set_trace()
---> 14             output[f"Observation_{hop+1}"] = "Failed to parse action. Bad formatting or incorrect action name."
     15 



--KeyboardInterrupt--

KeyboardInterrupt: Interrupted by user
> /tmp/ipykernel_2113022/1195860776.py(14)act()
     13             import ipdb; ipdb.set_trace()
---> 14             output[f"Observation_{hop+1}"] = "Failed to parse action. Bad formatting or incorrect action name."
     15 



In [85]:
llm.inspect_history(n=2)





You will be given `context`, `question`, `answer` and you will respond with `keywords`.

To do this, you will interleave Thought, Action, and Observation steps.

Thought can reason about the current situation, and Action can be the following types:

(1) Search[query], which takes a search query and returns one or more potentially relevant passages from a corpus
(2) Finish[keywords], which returns the final `keywords` and finishes the task

---

Follow the following format.

Context: may contain relevant facts

Question: ${question}

Thought 1: next steps to take based on last observation

Action 1: always either Search[query] or, when done, Finish[answer]

Observation 1: observations based on action

Thought 2: next steps to take based on last observation

Action 2: always either Search[query] or, when done, Finish[answer]

Observation 2: observations based on action

Thought 3: next steps to take based on last observation

Action 3: always either Search[query] or, when done, Finis

### Initialize DSpy program

In [90]:
uncompiled_rag = Digest()

### Testing inference before compiling

In [84]:
print(uncompiled_rag("Which genes are related to increased hemolymph glucose?").answer)

In the context provided, the genes GCK pLOF, GCK damaging missense, GIGYF1 pLOF, and G6PC2 damaging missense variants have been significantly associated with increased hemolymph glucose levels.


In [85]:
print(uncompiled_rag("Which genes are related to increased hemolymph glucose?").keywords)

MTNR1B | G6PC2 | GCK | PANK1 | Fgf15 | Gm17685 | Pla2g4a | Ptgs2 | ADCY5 | MADD


In [236]:
dev_set.passages[7].inputs().items()

[('question', 'List genes for diabetes susceptibility in BXD mice.')]

### Evaluate our RAG Program before it is compiled

This part is buggy due to the errors:

```
'str' object has no attribute 'inputs'
```

and 

```python
File /opt/fahamu/wolfshead/priv/ice/venv/lib/python3.10/site-packages/dspy/evaluate/evaluate.py:188, in merge_dicts(d1, d2)
    186 def merge_dicts(d1, d2):
    187     merged = {}
--> 188     for k, v in d1.items():
    189         if k in d2:
    190             merged[f"example_{k}"] = v

AttributeError: 'str' object has no attribute 'items'
```

Once it is fixed, continuing with this tutorial from section `4. DSPy Optimization`: [Getting started with RAG in DSPy](https://github.com/weaviate/recipes/blob/main/integrations/dspy/1.Getting-Started-with-RAG-in-DSPy.ipynb)

In [237]:
evaluate(Digest(), metric=llm_metric)

Average Metric: 0.0 / 1  (0.0): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1683.78it/s]

Error for example in dev set: 		 'str' object has no attribute 'inputs'
Average Metric: 0.0 / 1  (0.0%)


AttributeError: 'str' object has no attribute 'items'